In [1]:
import json, re
import numpy as np
import pandas as pd
from pathlib import Path

ANNOT = Path("linkedin-cvs-annotated.json")
UNLAB = Path("linkedin-cvs-not-annotated.json")

with open(ANNOT, "r", encoding="utf-8") as f:
    profiles_annot = json.load(f)

with open(UNLAB, "r", encoding="utf-8") as f:
    profiles_unlab = json.load(f)

print("annot type:", type(profiles_annot), "len:", len(profiles_annot))
print("unlab type:", type(profiles_unlab), "len:", len(profiles_unlab))

# if list-of-lists, flatten
if isinstance(profiles_annot, list) and len(profiles_annot)>0 and isinstance(profiles_annot[0], list):
    profiles_annot = [x for sub in profiles_annot for x in sub]
if isinstance(profiles_unlab, list) and len(profiles_unlab)>0 and isinstance(profiles_unlab[0], list):
    profiles_unlab = [x for sub in profiles_unlab for x in sub]

print("after flatten: annot len:", len(profiles_annot), "unlab len:", len(profiles_unlab))
print("top keys example:", list(profiles_annot[0].keys()))


annot type: <class 'list'> len: 609
unlab type: <class 'list'> len: 390
after flatten: annot len: 2638 unlab len: 1886
top keys example: ['organization', 'linkedin', 'position', 'startDate', 'endDate', 'status', 'department', 'seniority']


In [2]:
# DEBUG: inspect one profile keys and nested structure
p0 = profiles_unlab[0]
print("Top keys:", list(p0.keys()))

# print nested keys if there is "linkedin" dict
if isinstance(p0.get("linkedin"), dict):
    print("\nlinkedin keys:", list(p0["linkedin"].keys()))
    for k in ["headline", "summary", "about", "skills", "experience", "positions", "description"]:
        if k in p0["linkedin"]:
            print("FOUND in linkedin:", k, "->", type(p0["linkedin"][k]))
else:
    print("\nNo linkedin dict found. linkedin type:", type(p0.get("linkedin")))


Top keys: ['organization', 'linkedin', 'position', 'startDate', 'endDate', 'status']

No linkedin dict found. linkedin type: <class 'str'>


In [3]:
def clean_text(x):
    x = "" if x is None else str(x)
    return re.sub(r"\s+", " ", x).strip()

def build_text(row):
    # your schema: position, organization, linkedin, startDate, endDate, status
    pos = clean_text(row.get("position", ""))
    org = clean_text(row.get("organization", ""))
    status = clean_text(row.get("status", ""))
    sd = clean_text(row.get("startDate", ""))
    ed = clean_text(row.get("endDate", ""))

    link = row.get("linkedin", "")
    if isinstance(link, dict):
        link_txt = " ".join([clean_text(v) for v in link.values() if v is not None])
    else:
        link_txt = clean_text(link)

    parts = [pos, org, link_txt, sd, ed, status]
    return " | ".join([p for p in parts if p])

rows = []
for i, r in enumerate(profiles_annot):
    status = str(r.get("status", "")).upper()
    has_active = (status == "ACTIVE")

    rows.append({
        "idx": i,
        "text": build_text(r),
        "domain": r.get("department", None),
        "seniority": r.get("seniority", None),
        "has_active": has_active
    })

df = pd.DataFrame(rows)

df_train = df[
    df["has_active"] &
    df["text"].str.len().gt(0) &
    df["domain"].notna() &
    df["seniority"].notna()
].copy()

print("All:", len(df), "Trainable:", len(df_train))
print("Domains:", df_train["domain"].nunique(), "Seniorities:", df_train["seniority"].nunique())
df_train.head()


All: 2638 Trainable: 623
Domains: 11 Seniorities: 6


,idx,text,domain,seniority,has_active
0,0,Prokurist | Depot4Design GmbH | https://www.li...,Other,Management,True
1,1,CFO | Depot4Design GmbH | https://www.linkedin...,Other,Management,True
2,2,Betriebswirtin | Depot4Design GmbH | https://w...,Other,Professional,True
3,3,Prokuristin | Depot4Design GmbH | https://www....,Other,Management,True
4,4,CFO | Depot4Design GmbH | https://www.linkedin...,Other,Management,True


In [4]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

domain_labels = sorted(df_train["domain"].unique())
seniority_labels = sorted(df_train["seniority"].unique())

domain2id = {l:i for i,l in enumerate(domain_labels)}
id2domain = {i:l for l,i in domain2id.items()}

sen2id = {l:i for i,l in enumerate(seniority_labels)}
id2sen = {i:l for l,i in sen2id.items()}

df_train["domain_id"] = df_train["domain"].map(domain2id)
df_train["seniority_id"] = df_train["seniority"].map(sen2id)

train_df, val_df = train_test_split(
    df_train, test_size=0.2, random_state=42,
    stratify=df_train["domain_id"]
)

ds_dom_train = Dataset.from_pandas(train_df[["text","domain_id"]].rename(columns={"domain_id":"label"}))
ds_dom_val   = Dataset.from_pandas(val_df[["text","domain_id"]].rename(columns={"domain_id":"label"}))

ds_sen_train = Dataset.from_pandas(train_df[["text","seniority_id"]].rename(columns={"seniority_id":"label"}))
ds_sen_val   = Dataset.from_pandas(val_df[["text","seniority_id"]].rename(columns={"seniority_id":"label"}))

print(ds_dom_train, ds_sen_train)


Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 498
}) Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 498
})


In [5]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [6]:
import numpy as np
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

ds_dom_train = ds_dom_train.map(tokenize_batch, batched=True)
ds_dom_val   = ds_dom_val.map(tokenize_batch, batched=True)

ds_sen_train = ds_sen_train.map(tokenize_batch, batched=True)
ds_sen_val   = ds_sen_val.map(tokenize_batch, batched=True)

cols = ["input_ids", "attention_mask", "label"]
ds_dom_train.set_format(type="torch", columns=cols)
ds_dom_val.set_format(type="torch", columns=cols)
ds_sen_train.set_format(type="torch", columns=cols)
ds_sen_val.set_format(type="torch", columns=cols)

print("Tokenized ✔")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Tokenized ✔


In [7]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_domains = len(domain_labels)

dom_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_domains
)

args_dom = TrainingArguments(
    output_dir="./model_domain",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none",
)

trainer_dom = Trainer(
    model=dom_model,
    args=args_dom,
    train_dataset=ds_dom_train,
    eval_dataset=ds_dom_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_dom.train()
trainer_dom.evaluate()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2199441030.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_dom = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.659719,0.552000,0.064667
2,No log,1.608751,0.552000,0.064667
3,No log,1.581260,0.552000,0.064667


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.659719467163086,
 'eval_accuracy': 0.552,
 'eval_macro_f1': 0.06466729147141519,
 'eval_runtime': 26.1054,
 'eval_samples_per_second': 4.788,
 'eval_steps_per_second': 0.153,
 'epoch': 3.0}

In [9]:
num_sen = len(seniority_labels)

sen_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_sen
)

args_sen = TrainingArguments(
    output_dir="./model_seniority",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none",
)

trainer_sen = Trainer(
    model=sen_model,
    args=args_sen,
    train_dataset=ds_sen_train,
    eval_dataset=ds_sen_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_sen.train()
trainer_sen.evaluate()


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3722628873.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_sen = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.488309,0.480000,0.193789
2,No log,1.330510,0.552000,0.266918
3,No log,1.240708,0.592000,0.300782


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.240707516670227,
 'eval_accuracy': 0.592,
 'eval_macro_f1': 0.300781522117729,
 'eval_runtime': 25.9682,
 'eval_samples_per_second': 4.814,
 'eval_steps_per_second': 0.154,
 'epoch': 3.0}

In [10]:
import re
import pandas as pd

def clean_text(x):
    x = "" if x is None else str(x)
    return re.sub(r"\s+", " ", x).strip()

def build_text(row):
    def pick(*keys):
        for k in keys:
            v = row.get(k, None)
            if v is not None and str(v).strip() != "":
                return v
        return ""

    pos = clean_text(pick("position", "title"))
    org = clean_text(pick("organization", "company"))
    status = clean_text(pick("status", "job_status"))
    sd = clean_text(pick("startDate", "start_date"))
    ed = clean_text(pick("endDate", "end_date"))
    department = clean_text(pick("department", "domain"))

    link = row.get("linkedin", {})
    if not isinstance(link, dict):
        link = {}

    headline = clean_text(link.get("headline", ""))
    summary  = clean_text(link.get("summary", link.get("about", "")))

    # skills
    skills = link.get("skills", "")
    if isinstance(skills, list):
        skills_txt = " ".join(clean_text(s) for s in skills)
    else:
        skills_txt = clean_text(skills)

    # descriptions can be inside experiences/positions lists
    desc_txt = ""
    for key in ["experience", "positions", "jobs"]:
        v = link.get(key, None)
        if isinstance(v, list) and len(v) > 0:
            # concatenate first few descriptions
            descs = []
            for item in v[:3]:
                if isinstance(item, dict):
                    descs.append(clean_text(item.get("description", "")))
            desc_txt = " ".join([d for d in descs if d])
            if desc_txt:
                break

    parts = [pos, org, department, headline, summary, skills_txt, desc_txt, sd, ed, status]
    return " | ".join([p for p in parts if p])

rows_un = []
for i, r in enumerate(profiles_unlab):
    rows_un.append({
        "idx": i,
        "text": build_text(r),
        "status": str(r.get("status",""))
    })

df_unlab = pd.DataFrame(rows_un)
print(df_unlab.shape)
df_unlab.head()


(1886, 3)


,idx,text,status
0,0,"Bookkeeper | Keeping The Books, Bookkeeping | ...",ACTIVE
1,1,Co-Owner | Playful Paws | 2018-11 | ACTIVE,ACTIVE
2,2,Logistics Officer | S&R services | 2019-09 | 2...,INACTIVE
3,3,Truck driver/ laborer | ABC Supply Co. Inc. | ...,INACTIVE
4,4,Fuel Driver | MB Railways | 2018-03 | 2019-03 ...,INACTIVE


In [ ]:
import torch
from torch.utils.data import DataLoader
from datasets import Dataset

def tokenize_only(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

def predict_ids(trainer, texts, batch_size=32):
    ds = Dataset.from_dict({"text": texts})
    ds = ds.map(tokenize_only, batched=True)
    ds.set_format(type="torch", columns=["input_ids","attention_mask"])
    loader = DataLoader(ds, batch_size=batch_size)

    trainer.model.eval()
    device = trainer.model.device
    preds = []

    for batch in loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.no_grad():
            out = trainer.model(**batch)
            p = out.logits.argmax(dim=-1).cpu().numpy().tolist()
            preds.extend(p)
    return preds

dom_pred_ids = predict_ids(trainer_dom, df_unlab["text"].tolist())
sen_pred_ids = predict_ids(trainer_sen, df_unlab["text"].tolist())

df_unlab["pred_domain"] = [id2domain[i] for i in dom_pred_ids]
df_unlab["pred_seniority"] = [id2sen[i] for i in sen_pred_ids]

df_unlab.to_csv("predictions_finetuned_approach3.csv", index=False)
df_unlab.head()


Map:   0%|          | 0/1886 [00:00<?, ? examples/s]